In [13]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import warnings

warnings.filterwarnings('ignore')

In [26]:
def ucb_acquisition(x, gp, kappa = 2.0):
    #Upper Confidence Bound (UCB) acquisition function
    #x = x.reshape(1, -1)
    
    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    #ucb = mu + np.sqrt(beta) * sigma
    
    #return -ucb[0] # Return the negative UCB for minimisation
    return mu + kappa * sigma

def format_submission_string(x_best):
    x_formatted = [f"{coord:.6f}" for coord in x_best]
    return "-".join(x_formatted)

In [27]:
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy')
    
print(X)
print(Y)

[[0.32 0.76]
 [0.57 0.88]
 [0.73 0.73]
 [0.84 0.26]
 [0.65 0.68]
 [0.41 0.15]
 [0.31 0.08]
 [0.68 0.86]
 [0.08 0.40]
 [0.88 0.58]]
[0.00 0.00 0.00 0.00 -0.00 -0.00 -0.00 0.00 0.00 0.00]


In [ ]:
X_init = np.array([
[0.31940389, 0.76295937],
[0.57432921, 0.8798981 ],
[0.73102363, 0.73299988],
[0.84035342, 0.26473161],
[0.65011406, 0.68152635],
[0.41043714, 0.1475543 ],
[0.31269116, 0.07872278],
[0.68341817, 0.86105746],
[0.08250725, 0.40348751],
[0.88388983, 0.58225397]
])

y_init = np.array(
    [ 1.32267704e-079,  
    1.03307824e-046,
    7.71087511e-016,
    3.34177101e-124,
    -3.60606264e-003,
    -2.15924904e-054,
    -2.08909327e-091,
    2.53500115e-040,
    3.60677119e-081,
    6.22985647e-048]
)

y_trans = y_init.copy()

kernel = Matern(length_scale=0.1, length_scale_bounds=(1e-1, 1e2), nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)

gp.fit(X_init, y_trans)

grid_size = 100
x1 = np.linspace(0, 1, grid_size)
x2 = np.linspace(0, 1, grid_size)
X_candidates = np.array([[i, j] for i in x1 for j in x2])

# --- 5. Evaluate acquisition function ---
acq_values = ucb_acquisition(X_candidates, gp, kappa=2.5)

# --- 6. Select next point ---
next_point = X_candidates[np.argmax(acq_values)]

print(format_submission_string(next_point))
#print(f"Next point to query:{next_point:.6f}")

1.000000-0.000000
